In [59]:
import os
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [60]:
from langchain.agents import create_react_agent, AgentExecutor

In [78]:
# os.environ["SSL_CERT_FILE"] = certifi.where()
loaded = load_dotenv()
print("Loaded:", loaded)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHER_STACK_API")

Loaded: True


In [79]:
search_tool = TavilySearchResults(max_results=2)

In [80]:
# Custom Tool Function

@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a given city using WeatherStack API.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"
    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']} degree Celcius\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [81]:
result = search_tool.invoke("Give me the latest news on AI")
print(result)

[{'url': 'https://www.artificialintelligence-news.com', 'content': "Finance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely available\n\nAI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 2026\n\n### US and Japan announce sweeping AI and tech collaboration\n\nArtificial Intelligence\n\nApril 11, 2024\n\n### UK and Canada sign AI compute agreement\n\nArtificial Intelligence\n\nJanuary 31, 2024\n\n#### Machine Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026\n\n### Google folds Display Ads into AI-first Demand Gen platform\n\nMarketing AI\n\nMay 27, 2026\n\n### IDC: How EMEA CIOs can jumpstart AI rollouts\n\nGovernance, Regulation & Policy\n\nApril 29, 2026\n\n#### Enterprise [...] April 29, 2026\n\n#### Enterprise\n\n### How C3 AI agents will automate predictive maintenance for S

In [82]:
# LLM

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.9, openai_api_key=OPENAI_API_KEY)

In [83]:
# Prompt

prompt = hub.pull("hwchase17/react")
print(prompt)

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


/Users/harshsharma/Agentic AI Course/langchain_single_agent/.venv/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [84]:
# Tools

tools = [search_tool, get_weather_data]

In [85]:
# Create the ReAct agent

agent = create_react_agent(llm = llm, tools = tools, prompt = prompt)

In [86]:
# Agent Executor

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [87]:
# Run the agent with a query

response = agent_executor.invoke({
    "input": (
        "What is the capital of India and what is its current weather?"
    )
})



> Entering new AgentExecutor chain...
We need to find out the capital of India first, then check the current weather in that city.
Action: tavily_search_results_json
Action Input: "capital of India"[{'url': 'https://www.countryreports.org/country/India/capital-city.htm', 'content': '|  |  |\n --- |\n| Capital City | New Delhi |\n| Capital location | 28 36 N, 77 12 E |\n| Capital - history | the city\'s name is associated with various myths and legends; the original name for the city may have been Dhilli or Dhillika; alternatively, the name could be a corruption of the Hindustani words "dehleez" or "dehali" - both terms meaning "threshold" or "gateway" - and indicative of the city as a gateway to the Gangetic Plain; after the British decided to move the capital of their Indian Empire from Calcutta to Delhi in 1911, they created a new governmental district south of the latter designated as New Delhi; the new capital was not formally inaugurated until 1931. |\n| Capital - time differenc

In [56]:
print(response["output"])

AI is expected to transform many cognitive tasks across industries, creating new efficiencies and roles while complementing human expertise in decision-making and problem-solving. It is essential for individuals and organizations to invest in developing the necessary skills for an AI-augmented workplace to ensure economic growth and corporate differentiation.
